In [ ]:
# Install all the dependencies
!pip install transformers
!pip install torch torchvision torchaudio
!pip install stable-diffusion
!pip install git+https://github.com/openai/CLIP.git
!pip install moviepy
!pip install pillow

  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-6yga9ogo
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-6yga9ogo
  Resolved https://github.com/openai/CLIP.git to commit dcba3cb2e2827b402d2701e7e1c7d9fed8a20ef1
  Preparing metadata (setup.py) ... done


In [ ]:
from diffusers import StableDiffusionPipeline
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

# Load the StableDiffusion Text2Img model
stable_diff_pipe = StableDiffusionPipeline.from_pretrained("CompVis/stable-diffusion-v1-4", torch_dtype=torch.float16)
stable_diff_pipe.to(device)

# Generate a picture from a prompt
def generate_image(prompt):
    image = stable_diff_pipe(prompt).images[0]
    return image


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

In [ ]:
# Poem exemple
poem = [
    "Under the glow of a lonely moon,"
    "Trees dance to the rhythm of the wind.",
    "The sun shines in my life,",
    "And yet I walk in the rain,",
    "rain that saws me, bends me, empties me,",
    "humic weather that tires and wrinkles me."
]

# Generate a picture to each line
base_images = []
for line in poem:
    img = generate_image(line)
    base_images.append(img)


  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

In [ ]:
import numpy as np

# Convert our list to a np.array
image_arrays = [np.array(img) for img in base_images]

In [ ]:
from diffusers import StableDiffusionImg2ImgPipeline
from PIL import Image
import torch

# Load the StableDiffusion Img2Img model
pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
    "CompVis/stable-diffusion-v1-4", torch_dtype=torch.float16
).to(device)


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

In [ ]:
from PIL import Image
import numpy as np
from moviepy.editor import ImageSequenceClip
from tqdm import tqdm

# Parameters
nb_steps = 15  # number of intermediate frames per transition
image_size = (512, 512)
strength = 0.6
guidance_scale = 7.5

# Text blending function
def blend_prompts(prompt1, prompt2, alpha):
    return f"{(1 - alpha):.2f} {prompt1}, {(alpha):.2f} {prompt2}"

In [ ]:
# Initialisation
all_frames = []

# Start image = image of first line
current_image = generate_image(poem[0]).resize(image_size).convert("RGB")
all_frames.append(np.array(current_image))

# Iterate through the elements of the poem, comparing each element with the next one
for i in tqdm(range(len(poem) - 1)):
    # Define the two prompts to blend: the current element and the next element in the poem
    prompt1 = poem[i]
    prompt2 = poem[i + 1]

    # Perform 'nb_steps' transitions between the two prompts
    for step in range(nb_steps):
        # Calculate alpha to interpolate between the two prompts, ranging from 0 ('closer' to prompt1) to 1 ('closer' to prompt2)
        alpha = step / (nb_steps - 1)

        # Generate a blended prompt based on the calculated alpha
        blended_prompt = blend_prompts(prompt1, prompt2, alpha)

        # Generate the image using the `pipe` function with the blended prompt
        # `current_image` is the starting image (or the resulting image from the previous step to always keep a fluid animation)
        result = pipe(
            prompt=blended_prompt,
            image=current_image,
            strength=strength,
            guidance_scale=guidance_scale
        ).images[0]

        current_image = result.resize(image_size).convert("RGB")
        all_frames.append(np.array(current_image))


  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

...

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

100%|██████████| 5/5 [06:15<00:00, 75.02s/it]


In [ ]:
from moviepy.editor import ImageSequenceClip

# Generate the animation
clip = ImageSequenceClip(all_frames, fps=6)
clip.write_videofile("video.mp4", codec="libx264")


Moviepy - Building video morphing_poem.mp4.
Moviepy - Writing video morphing_poem.mp4



Moviepy - Done !
Moviepy - video ready morphing_poem.mp4
